In [1]:
import pandas as pd, numpy as np, json, re
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report

D     = r"C:\Users\Bhasura\Desktop\4th year Research\dataset research\com 2\archive2\Edge-IIoTset dataset"        # ඔයාගේ dataset folder path එක
TRAIN = D + r"\EdgeIIoT_train_80.csv"
TEST  = D + r"\EdgeIIoT_test_20.csv"
FULL  = D + r"\ML-EdgeIIoT-dataset.csv"

tr   = pd.read_csv(TRAIN, low_memory=False)
te   = pd.read_csv(TEST,  low_memory=False)
full = pd.read_csv(FULL,  low_memory=False)
print(tr.shape, te.shape, full.shape)       # (126240, 62) (31560, 62) (157800, 63)


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Bhasura\\Desktop\\4th year Research\\dataset research\\com 2\\archive2\\Edge-IIoTset dataset\\EdgeIIoT_train_80.csv'

In [2]:
import pandas as pd, numpy as np, json, re
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report

D     = r"C:\Users\Bhasura\Desktop\4th year Research\dataset research\com 2\archive2\Edge-IIoTset dataset\Selected dataset for ML and DL"        # ඔයාගේ dataset folder path එක
TRAIN = D + r"\EdgeIIoT_train_80.csv"
TEST  = D + r"\EdgeIIoT_test_20.csv"
FULL  = D + r"\ML-EdgeIIoT-dataset.csv"

tr   = pd.read_csv(TRAIN, low_memory=False)
te   = pd.read_csv(TEST,  low_memory=False)
full = pd.read_csv(FULL,  low_memory=False)
print(tr.shape, te.shape, full.shape)       # (126240, 62) (31560, 62) (157800, 63)

(126240, 62) (31560, 62) (157800, 63)


In [3]:
cols = list(tr.columns)

def norm(df):
    out = {}
    for c in cols:
        s = df[c]
        n = pd.to_numeric(s, errors='coerce')
        txt = s.astype(str).str.strip()
        out[c] = np.where(n.notna(), n.astype(float).astype(str), txt)
    return pd.DataFrame(out, index=df.index)

fs = norm(full)
fs['Attack_type'] = full['Attack_type'].astype(str).str.strip().values
fs = fs.drop_duplicates()
amb = fs.duplicated(subset=cols, keep=False)
print("ambiguous feature-rows:", amb.sum())
fs = fs[~amb]

for name, d in [('train', tr), ('test', te)]:
    m = norm(d).merge(fs, on=cols, how='left')
    assert len(m) == len(d)
    d['Attack_type'] = m['Attack_type'].values
    print(name, "unlabeled:", d['Attack_type'].isna().sum())     # 0 වෙන්න ඕන

ambiguous feature-rows: 0
train unlabeled: 0
test unlabeled: 0


In [4]:
drop_cols = ['frame.time','ip.src_host','ip.dst_host','arp.dst.proto_ipv4',
             'arp.src.proto_ipv4','http.file_data','http.request.uri.query',
             'http.referer','http.request.full_uri','tcp.payload','mqtt.msg',
             'Attack_label','Attack_type']
X_tr = tr.drop(columns=drop_cols, errors='ignore').copy()
X_te = te.drop(columns=drop_cols, errors='ignore').copy()

new_tr, new_te = {}, {}
for c in list(X_tr.columns):
    if pd.api.types.is_numeric_dtype(X_tr[c]):
        X_tr[c] = X_tr[c].fillna(0); X_te[c] = X_te[c].fillna(0); continue
    a = pd.to_numeric(X_tr[c], errors='coerce')
    if a.notna().mean() > 0.99:
        X_tr[c] = a.fillna(0); X_te[c] = pd.to_numeric(X_te[c], errors='coerce').fillna(0)
    else:
        s_tr = X_tr[c].astype(str).str.strip(); s_te = X_te[c].astype(str).str.strip()
        cats = sorted(s_tr.unique())
        if len(cats) > 50:
            mp = {v: i for i, v in enumerate(cats)}
            X_tr[c] = s_tr.map(mp); X_te[c] = s_te.map(mp).fillna(-1)
        else:
            for k in cats:
                new_tr[f'{c}_{k}'] = (s_tr == k).astype(int)
                new_te[f'{c}_{k}'] = (s_te == k).astype(int)
            X_tr = X_tr.drop(columns=c); X_te = X_te.drop(columns=c)
X_tr = pd.concat([X_tr, pd.DataFrame(new_tr, index=X_tr.index)], axis=1)
X_te = pd.concat([X_te, pd.DataFrame(new_te, index=X_te.index)], axis=1)
X_te = X_te.reindex(columns=X_tr.columns, fill_value=0)

# payload / URL fields: length + ID (train එකෙන් විතරක් හදපු maps)
txt_cols = ['http.file_data','http.request.uri.query','http.referer',
            'http.request.full_uri','tcp.payload','mqtt.msg']
maps = {}
for c in txt_cols:
    s = tr[c].astype(str).str.strip(); vc = s.value_counts()
    maps[c] = {v: i for i, v in enumerate(vc[vc >= 5].index)}

def add_txt(X, src):
    X = X.copy()
    for c in txt_cols:
        s = src[c].astype(str).str.strip()
        X[f'{c}_len']  = s.str.len().values
        X[f'{c}_code'] = s.map(maps[c]).fillna(-1).values
    return X

X_tr2, X_te2 = add_txt(X_tr, tr), add_txt(X_te, te)

# XGBoost වලට [ ] < අකුරු තියෙන column නම් බෑ
seen, newcols = {}, []
for c in X_tr2.columns:
    n = re.sub(r'[\[\]<>]', '_', str(c))
    if n in seen: seen[n] += 1; n = f"{n}__{seen[n]}"
    else: seen[n] = 0
    newcols.append(n)
X_tr2.columns = newcols; X_te2.columns = newcols
print("features:", X_tr2.shape[1])             # 71 වෙන්න ඕන

ok = tr['Attack_type'].notna().values
le = LabelEncoder().fit(tr.loc[ok, 'Attack_type'])
y_tr = le.transform(tr.loc[ok, 'Attack_type'])

model2 = XGBClassifier(n_estimators=100, objective='multi:softprob',
                       tree_method='hist', n_jobs=-1, random_state=42)
model2.fit(X_tr2[ok], y_tr)

proba2 = model2.predict_proba(X_te2)
out2 = pd.DataFrame({'row': np.arange(len(te)),
                     'predicted_type': le.inverse_transform(proba2.argmax(axis=1)),
                     'confidence': proba2.max(axis=1).round(6)})
out2.to_csv('xgb_predictions_test20_v2.csv', index=False)

lab = te['Attack_type'].notna().values
yt, yp = te.loc[lab, 'Attack_type'].values, out2.loc[lab, 'predicted_type'].values
print("accuracy :", accuracy_score(yt, yp))
print("macro-F1 :", f1_score(yt, yp, average='macro', labels=sorted(set(yt))))
json.dump(list(X_tr2.columns), open('feature_names_15class_v2.json', 'w'))

features: 71
accuracy : 0.9064955640050697
macro-F1 : 0.8896306317494371


In [5]:
bad = {"0", "0.0", "nan"}
al = pd.DataFrame({
    "flow_id": np.arange(len(te)),
    "src_ip": te["ip.src_host"].astype(str).values,
    "dst_ip": te["ip.dst_host"].astype(str).values,
    "predicted_type": out2["predicted_type"].values,
    "confidence": out2["confidence"].values,
    "event_time": te["frame.time"].astype(str).values,
})
ok_ip = ~al.src_ip.isin(bad) & ~al.dst_ip.isin(bad)

normal  = al[ok_ip & (al.predicted_type == "Normal")].sample(2, random_state=1)
attacks = al[ok_ip & (al.predicted_type != "Normal")].groupby("predicted_type").head(1).sample(3, random_state=1)
pick = pd.concat([normal, attacks]).sort_values("flow_id")

pick.to_csv("sample_alerts.csv", index=False)
pick.to_json("sample_alerts.json", orient="records", indent=2)
print(pick.to_csv(index=False))

flow_id,src_ip,dst_ip,predicted_type,confidence,event_time
6,192.168.0.170,192.168.0.128,Ransomware,0.965039, 2021 22:49:49.709983000 
7,192.168.0.128,192.168.0.170,DDoS_HTTP,0.99404, 2021 11:36:50.031957000 
8,192.168.0.170,192.168.0.128,Backdoor,0.998752, 2021 20:59:13.616844000 
8509,192.168.0.101,192.168.0.128,Normal,0.999962, 2021 18:20:11.441842000 
28500,192.168.0.101,192.168.0.128,Normal,0.99999, 2021 21:55:30.312224000 



In [6]:
import time, os, joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_fscore_support

lab = te['Attack_type'].notna().values
yt  = te.loc[lab, 'Attack_type'].values

def run(name, mdl):
    t0 = time.perf_counter(); mdl.fit(X_tr2[ok], y_tr); train_s = time.perf_counter() - t0
    t0 = time.perf_counter(); proba = mdl.predict_proba(X_te2); lat = (time.perf_counter() - t0) / len(X_te2) * 1000
    yp = le.inverse_transform(proba.argmax(axis=1))[lab]
    joblib.dump(mdl, f"{name}_tmp.joblib"); size = os.path.getsize(f"{name}_tmp.joblib") / 1024 / 1024
    os.remove(f"{name}_tmp.joblib")
    print(f"{name:12s} acc={accuracy_score(yt, yp):.4f}  macroF1={f1_score(yt, yp, average='macro', labels=sorted(set(yt))):.4f}  "
          f"train={train_s:.1f}s  latency={lat:.4f} ms/sample  size={size:.2f} MB")
    return yp

yp_rf  = run("RandomForest", RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42))
yp_xgb = run("XGBoost", XGBClassifier(n_estimators=100, objective='multi:softprob',
                                      tree_method='hist', n_jobs=-1, random_state=42))

labels = sorted(set(yt))
f_rf  = precision_recall_fscore_support(yt, yp_rf,  labels=labels, zero_division=0)[2]
f_xgb = precision_recall_fscore_support(yt, yp_xgb, labels=labels, zero_division=0)[2]
print(pd.DataFrame({"RF_F1": f_rf.round(3), "XGB_F1": f_xgb.round(3)}, index=labels))

RandomForest acc=0.9369  macroF1=0.9150  train=4.1s  latency=0.0087 ms/sample  size=8.15 MB
XGBoost      acc=0.9065  macroF1=0.8896  train=11.2s  latency=0.0035 ms/sample  size=1.35 MB
                       RF_F1  XGB_F1
Backdoor               0.951   0.811
DDoS_HTTP              0.699   0.637
DDoS_ICMP              1.000   1.000
DDoS_TCP               1.000   1.000
DDoS_UDP               1.000   1.000
Fingerprinting         0.904   0.902
MITM                   1.000   1.000
Normal                 1.000   1.000
Password               0.225   0.260
Port_Scanning          0.992   0.992
Ransomware             0.953   0.745
SQL_injection          1.000   1.000
Uploading              1.000   1.000
Vulnerability_scanner  1.000   0.998
XSS                    1.000   1.000


In [8]:
TIME_RE = r"\d{4} \d{2}:\d{2}:\d{2}\.\d{9}"
vtr = tr["frame.time"].astype(str).str.contains(TIME_RE).values       # True = සාමාන්‍ය row
vte = te["frame.time"].astype(str).str.contains(TIME_RE).values

# 1) Class එකක් අනුව shifted rows
tab = pd.DataFrame({"cls": tr["Attack_type"], "shifted": ~vtr}).groupby("cls")["shifted"].agg(["sum", "count"])
tab["share"] = (tab["sum"] / tab["count"]).round(3)
print(tab)

# 2) Train: සියලු rows vs valid rows පමණක්. Test: valid rows vs shifted rows
def score(y, p): return accuracy_score(y, p), f1_score(y, p, average='macro', labels=sorted(set(y)))
makers = {"RandomForest": lambda: RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42),
          "XGBoost": lambda: XGBClassifier(n_estimators=100, objective='multi:softprob',
                                           tree_method='hist', n_jobs=-1, random_state=42)}
res = []
for name, mk in makers.items():
    for tname, tmask in [("train: all rows", np.ones(len(tr), bool)), ("train: valid only", vtr)]:
        m = tmask & ok
        assert tr.loc[m, 'Attack_type'].nunique() == 15, "A class is missing from training"
        mdl = mk(); mdl.fit(X_tr2[m], le.transform(tr.loc[m, 'Attack_type']))
        pred = le.inverse_transform(mdl.predict(X_te2))
        for sname, emask in [("test: valid", vte & lab), ("test: shifted", ~vte & lab)]:
            a, f = score(te.loc[emask, 'Attack_type'].values, pred[emask])
            res.append((name, tname, sname, int(emask.sum()), round(a, 4), round(f, 4)))
print(pd.DataFrame(res, columns=["model", "training", "evaluated on", "n_rows", "accuracy", "macroF1"]).to_string(index=False))

                         sum  count  share
cls                                       
Backdoor                   0   8121    0.0
DDoS_HTTP                  0   8526    0.0
DDoS_ICMP                  0  11357    0.0
DDoS_TCP                   0   8261    0.0
DDoS_UDP               11534  11534    1.0
Fingerprinting             0    806    0.0
MITM                     978    978    1.0
Normal                     0  19441    0.0
Password                   0   7986    0.0
Port_Scanning              0   8015    0.0
Ransomware                 0   8658    0.0
SQL_injection              0   8264    0.0
Uploading                  0   8173    0.0
Vulnerability_scanner      0   8044    0.0
XSS                        0   8076    0.0


AssertionError: A class is missing from training

In [9]:
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score

TIME_RE = r"\d{4} \d{2}:\d{2}:\d{2}\.\d{9}"
vtr = tr["frame.time"].astype(str).str.contains(TIME_RE).values
vte = te["frame.time"].astype(str).str.contains(TIME_RE).values

# test set eke shifted share per class
print(pd.DataFrame({"cls": te["Attack_type"], "shifted": ~vte})
        .groupby("cls")["shifted"].agg(["sum","count"]).query("sum>0"))

keep = sorted(tr.loc[vtr, "Attack_type"].unique())      # 13 classes
print(len(keep), "classes survive valid-only:", keep)
le13 = LabelEncoder().fit(keep)

ytr_all = le.transform(tr["Attack_type"])
m_te    = vte & te["Attack_type"].isin(keep).values     # evaluate ONLY on the 13 clean classes
yte_str = te.loc[m_te, "Attack_type"].values

rows = []
for name, mk in makers.items():
    # A: train on all rows (15 classes), as before
    mA = mk(); mA.fit(X_tr2, ytr_all)
    pA = le.inverse_transform(np.asarray(mA.predict(X_te2[m_te])).astype(int))
    # B: train on valid rows only (13 classes)
    mB = mk(); mB.fit(X_tr2[vtr], le13.transform(tr.loc[vtr, "Attack_type"]))
    pB = le13.inverse_transform(np.asarray(mB.predict(X_te2[m_te])).astype(int))
    for tag, p in [("A: train all rows", pA), ("B: train valid-only", pB)]:
        rows.append([name, tag,
                     accuracy_score(yte_str, p),
                     f1_score(yte_str, p, labels=keep, average="macro")])

print(pd.DataFrame(rows, columns=["model","training","acc","macroF1(13 clean classes)"]).round(4).to_string(index=False))

NumExprClobberingError: Variables in expression "(sum) > (0)" overlap with builtins: ('sum')

In [10]:
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score

TIME_RE = r"\d{4} \d{2}:\d{2}:\d{2}\.\d{9}"
vtr = tr["frame.time"].astype(str).str.contains(TIME_RE).values
vte = te["frame.time"].astype(str).str.contains(TIME_RE).values

# shifted rows per class in the test set
t = (pd.DataFrame({"cls": te["Attack_type"], "shifted": ~vte})
       .groupby("cls")["shifted"].agg(["sum","count"]))
print(t[t["sum"] > 0])

keep = sorted(tr.loc[vtr, "Attack_type"].unique())      # 13 classes
print(len(keep), "classes survive valid-only:", keep)
le13 = LabelEncoder().fit(keep)
# ... (the rest: ytr_all, m_te, loop over makers, results table)

           sum  count
cls                  
DDoS_UDP  2964   2964
MITM       236    236
13 classes survive valid-only: ['Backdoor', 'DDoS_HTTP', 'DDoS_ICMP', 'DDoS_TCP', 'Fingerprinting', 'Normal', 'Password', 'Port_Scanning', 'Ransomware', 'SQL_injection', 'Uploading', 'Vulnerability_scanner', 'XSS']


In [11]:
rom sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score

TIME_RE = r"\d{4} \d{2}:\d{2}:\d{2}\.\d{9}"
vtr = tr["frame.time"].astype(str).str.contains(TIME_RE).values
vte = te["frame.time"].astype(str).str.contains(TIME_RE).values

# shifted rows per class in the test set
t = (pd.DataFrame({"cls": te["Attack_type"], "shifted": ~vte})
       .groupby("cls")["shifted"].agg(["sum","count"]))
print(t[t["sum"] > 0])

keep = sorted(tr.loc[vtr, "Attack_type"].unique())      # 13 classes
print(len(keep), "classes survive valid-only:", keep)
le13 = LabelEncoder().fit(keep)

SyntaxError: invalid syntax (1018981083.py, line 1)

In [13]:
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score

TIME_RE = r"\d{4} \d{2}:\d{2}:\d{2}\.\d{9}"
vtr = tr["frame.time"].astype(str).str.contains(TIME_RE).values
vte = te["frame.time"].astype(str).str.contains(TIME_RE).values

# shifted rows per class in the test set
t = (pd.DataFrame({"cls": te["Attack_type"], "shifted": ~vte})
       .groupby("cls")["shifted"].agg(["sum","count"]))
print(t[t["sum"] > 0])

keep = sorted(tr.loc[vtr, "Attack_type"].unique())      # 13 classes
print(len(keep), "classes survive valid-only:", keep)
le13 = LabelEncoder().fit(keep)
# ... (the rest: ytr_all, m_te, loop over makers, results table)
ytr_all = le.transform(tr["Attack_type"])
m_te    = vte & te["Attack_type"].isin(keep).values     # evaluate ONLY on the 13 clean classes
yte_str = te.loc[m_te, "Attack_type"].values

rows = []
for name, mk in makers.items():
    # A: train on all rows (15 classes), as before
    mA = mk(); mA.fit(X_tr2, ytr_all)
    pA = le.inverse_transform(np.asarray(mA.predict(X_te2[m_te])).astype(int))
    # B: train on valid rows only (13 classes)
    mB = mk(); mB.fit(X_tr2[vtr], le13.transform(tr.loc[vtr, "Attack_type"]))
    pB = le13.inverse_transform(np.asarray(mB.predict(X_te2[m_te])).astype(int))
    for tag, p in [("A: train all rows", pA), ("B: train valid-only", pB)]:
        rows.append([name, tag,
                     accuracy_score(yte_str, p),
                     f1_score(yte_str, p, labels=keep, average="macro")])

print(pd.DataFrame(rows, columns=["model","training","acc","macroF1(13 clean classes)"]).round(4).to_string(index=False))

           sum  count
cls                  
DDoS_UDP  2964   2964
MITM       236    236
13 classes survive valid-only: ['Backdoor', 'DDoS_HTTP', 'DDoS_ICMP', 'DDoS_TCP', 'Fingerprinting', 'Normal', 'Password', 'Port_Scanning', 'Ransomware', 'SQL_injection', 'Uploading', 'Vulnerability_scanner', 'XSS']
       model            training    acc  macroF1(13 clean classes)
RandomForest   A: train all rows 0.9298                     0.9020
RandomForest B: train valid-only 0.9316                     0.9039
     XGBoost   A: train all rows 0.8959                     0.8727
     XGBoost B: train valid-only 0.8917                     0.8679


In [14]:
import time
rng = np.random.default_rng(42)
ytr_all = le.transform(tr["Attack_type"])
idx = rng.choice(len(X_te2), 2000, replace=False)
is_df = hasattr(X_te2, "iloc")
row = (lambda i: X_te2.iloc[[idx[i]]]) if is_df else (lambda i: X_te2[idx[i]:idx[i]+1])
ytrue = te["Attack_type"].values[idx]

out = []
for name, mk in makers.items():
    m = mk(); m.fit(X_tr2, ytr_all)
    try: m.set_params(n_jobs=1)          # single-sample, no thread overhead
    except Exception: pass
    for i in range(20): m.predict(row(i))   # warm-up
    lat, pred = [], []
    for i in range(len(idx)):
        r = row(i)
        t0 = time.perf_counter()
        p = m.predict(r)[0]
        lat.append((time.perf_counter() - t0) * 1000)
        pred.append(le.inverse_transform([int(p)])[0])
    lat = np.array(lat); pred = np.array(pred)
    alerts = (pred != "Normal").sum()
    out.append([name, np.percentile(lat,50), np.percentile(lat,95), np.percentile(lat,99),
                1000/lat.mean(), (pred==ytrue).mean(), alerts])

print(pd.DataFrame(out, columns=["model","p50_ms","p95_ms","p99_ms","rows/sec","stream_acc","alerts_raised"]).round(3).to_string(index=False))

       model  p50_ms  p95_ms  p99_ms  rows/sec  stream_acc  alerts_raised
RandomForest   9.344  12.323  15.148   103.396       0.936           1711
     XGBoost   5.491  10.856  16.948   140.094       0.902           1711
